# Team and Project Meta Information

### Phase Leader

Daniel Yi (daniel_yi@berkeley.edu)

### Project Title

Turning Airline Delay Prediction into Operational Efficiency

### Group Number

Team 3_2

### Team Members

| Photo                                                                                                                                | Name        | Email                           |
|--------------------------------------------------------------------------------------------------------------------------------------|-------------|---------------------------------|
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/daniel.png" width="100">  | Daniel Yi   | daniel_yi@berkeley.edu          |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/frank.png" width="100">   | Frank Lin   | frank_tingchun_lin@berkeley.edu |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/jason.png" width="100">   | Jason Chang | chan572@berkeley.edu            |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/william.png" width="100"> | William Shu | william.shu@berkeley.edu        |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/allen.png" width="100">   | Allen Li    | allen_li@berkeley.edu           |

### Phase Leader Plan

| **Project Phase** | **Leader(s)**             | **Email**                                        |
|-------------------|---------------------------|--------------------------------------------------|
| Week 1 (Phase 1)  | Frank Lin                 | frank_tingchun_lin@berkeley.edu                  |
| Week 2 (Phase 2)  | William Shu               | william.shu@berkeley.edu                         |
| Week 3 (Phase 2)  | Daniel Yi                 | daniel_yi@berkeley.edu                           |
| Week 4 (Phase 3)  | Allen Li                  | allen_li@berkeley.edu                            |
| Week 5 (Phase 3)  | Jason Chang               | chan572@berkeley.edu                             |

# Project Abstract

Flight delays create cascading operational disruptions for airlines, impacting crew scheduling, aircraft rotations, and passenger connections, ultimately increasing costs and reducing customer satisfaction. In this project, we focus on the XYZ Airline’s Operations Control (OCC) team as the primary stakeholder, where early and accurate delay predictions enable proactive decision-making. We formulate the task as a binary classification problem, predicting whether a flight will experience a departure delay of 15 minutes or more, two hours before scheduled departure, using only information available at that time.

We use an integrated dataset combining _U.S. Department of Transportation flight performance data_ with _NOAA weather observations_, along with airport and temporal features. A Logistic Regression model is selected as the baseline due to its interpretability and role as a standard benchmark. However, given the strong class imbalance (~80% on-time vs 20% delayed) and the higher operational cost of missed delays, we prioritize Recall and F2-score as primary evaluation metrics. Our results show that while Logistic Regression is unstable across time, Random Forest consistently achieves stronger and more stable recall performance, demonstrating its ability to capture non-linear relationships in delay behavior. Key predictive features include weather conditions (precipitation, visibility, wind), temporal patterns (departure hour, seasonality), and operational factors (carrier and origin airport).

In this phase, we developed a scalable modeling pipeline, implemented imbalance-aware techniques, and validated a time-based cross-validation framework to prevent data leakage. Our findings confirm that predicting delays two hours in advance is both feasible and operationally actionable. In Phase III, we will focus on model optimization, including neural networks, enhanced feature engineering, and hyperparameter tuning, with the goal of delivering a production-ready model that aligns with airline operational decision-making.

# Project Description

#### Task to Tackle
We frame flight delay predictions as a binary classification task to determine two hours before scheduled departure whether a flight will experience a departure delay of at least 15 minutes or greater. Specifically, our labels will be:
1. **Delay** (≥15 min)
2. **No Delay** (< 15 min)

Furthermore, our primary clients will be the airlines, which rely on accurate early predictions to make informed operational decisions that directly affect efficiency and customer satisfaction.

#### Dataset Description

_NOTE: need to be exapanded to include a summary table view of all the datasets with the info including rows, columns, size, data link location, and missing values (?)_

To carry out our flight delay predictions, we use the On‑Time Flight Performance and Weather Dataset (abbreviated as OTPW). This dataset spans from 2015-2021 and combines hourly weather observations from the National Oceanic and Atmospheric Administration (NOAA), airport metadata from the Department of Transportation, and on‑time performance records for US flights from the Bureau of Transportation Statistics. We use a 1-year subset of this data from January 1 to December 31, 2015 which has dimensions `11,623,708 rows x 216 columns` and is `2.00 GB` in size. However, after deduplication, the dimensions reduce to `5,811,854 rows x 216 columns` with size `1.04 GB`. Each row represents a single flight paired with its origin and destination metadata, the corresponding hourly weather conditions, and the flight's recorded departure delay. 
The detailed data dictionary for this table is introduced in the following EDA section.



# EDA


### Data Dictionary of Raw Columns in 1y OTPW dataset (Used to inform feature selection/engineering)

| **Feature**              | **Description**                               | **Type**  | **Example(s)**                   |
|-------------             |-----------------------------                  |-----------|----------------------------------|
| FL_DATE                  | Date of flight                                | String    | 2015-10-25                 |
| OP_CARRIER               | Operating airline carrier code                | String    | DL                         |
| ORIGIN                   | Origin airport code                           | String    | DFW                        |
| DEST                     | Destination airport code                      | String    | ATL                        |
| DEP_DEL_15               | Departure delay by 15 min or greater (target) | Binary    | 0.0, 1.0                   |
| HourlyDryBulbTemperature | Dry-bulb temperature, commonly used as the standard air temperature reported (in Fahrenheit) | Integer | 43 |
| HourlyPrecipitation      | The amount of precipitation measured in millimeters | Double | 0.03 
| HourlyRelativeHumidity   | Relative humidity given to the nearest whole percetange between 0 and 100 | Integer | 45
| HourlyVisibility         | The horizontal distance an object can be seen and identified measured in miles | Double | 9.94
| HourlyWindSpeed          | Wind Speed (measured in miles per hour)       | Integer   | 6


TODO: 
-- Summary statistics (Use Overview, Cleaning, and Dataset selection/Imbalance Target slides for reference)

-- Correlation analysis (Spearman's correlation for matrix?)

-- Other useful text-based analysis


<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/spearman.png" width="600"><br>

### Feature Engineering

####  Missing data for Response Variable

As mentioned in the EDA, we are interested in whether a flight is delayed over 15 minutes. Our reponse variable representing flight delays from our one-year OTPW dataset contains 1.54% null values, all from cancelled flights. After removing these cancelled flights, we have 5,722,067 rows remaining in our dataset, with no more missing values in the delay response variable.

#### Other Missing data Explaination

A significant amount of missing data comes from the weather categories. However, we would like to use this data since we believe weather is an important predictor to weather or not a flight is delayed. Specifically, we are concerned with hourly weather categories.

One of the reasons why there are so many missing values is that the weather dataset has multiple report types (`FM-15`, `FM-16`, `FM-12`, etc) that do not contain all information. For example:

* `FM-15` represents standard hourly temperature, wind, visibility, pressure, and precipitation observations. This is the most complete and consistent report type, making it ideal for linking to individual flights.
* `FM-16` represents special weather reports that are issued when conditions change significantly (e.g., sudden storms or extreme winds). It is irregular and event-driven, so coverage is sparse.
* `FM-12` represents synoptic reports issued every 6–12 hours with fewer variables. Many fields are missing and it does not align well with hourly flight data.
* `SOD` (Summary of Day) consists of daily aggregates like total precipitation or max/min temperature; not useful for per-flight modeling.
* `SHEF` and `SY-MT` are rare or specialized reports (hydrology or system metadata) with minimal rows and missing data.

The one-year OTPW dataset has the following distribution of report types:

| Report Type | count   |
|-------------|---------|
| SHEF        | 999     |
| FM-15       | 5005738 |
| SOD         | 11952   |
| SY-MT       | 2504    |
| FM-12       | 228622  |
| FM-16       | 472252  |

Because of this, we focus primarily on `FM-15` for modeling. In the future we can optionally include `FM-16` if we want to capture extreme weather events. As for the other report types, we will drop them to ensure consistent and reliable features for each flight.

After filtering for only `FM-15` report types, there are 5,005,738 rows remaining.

#### Normalization and Transformations

Based off of our EDA, we can see that precipitation, humidity, wind speed, visibility and temperature are the strongest flight delay signals. We will use them in our modeling, and they will need to be normalized and transformed.

| Before | After |
|--------|-------|
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/precip.png" width="500"><br> | <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/precip_binary.png" width="468"><br> |
(In the "After" image showing binary distribution of precipitation, can we include percentage labels for each category?)

Precipitation is heavily skewed, even after log transformation. A better option would be to transform it into a binary feature where `0 = no precipitation` and `1 = precipitation`.

Next, we look at the other numeric features: humidity, wind speed, visibility and temperature. Relative humidity and temperature seem to be relatively normal, so we normalize them for use in ML modeling.

<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/humidity_temp_normalized.png" width="900"><br>

Wind speed is right skewed. We will log transform it and then scale it.

<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/wind_log_normalized.png" width="500"><br>

Visibility is sparse and left skewed, with a max. We decide to transform it into a multi-categorical variable instead.

| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

<br><img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/visibility_cat.png" width="500"><br> 

(can we include percentage labels for each visibility category?)

#### Feature Creation

Alongside the existing features, we will create two additional features to capture temporal and route-specific patterns in flight delays. First, we introduce a binary indicator for whether a flight occurs on a weekend (Saturday or Sunday). Travel patterns and airport congestion often differ between weekdays and weekends, which can influence delay behavior, making this a useful feature for the model.

Second, we construct a combined feature representing the route by concatenating the origin and destination airports. This allows the model to capture route-specific effects, as certain routes may consistently experience higher delays due to factors such as traffic volume, weather patterns, or airport efficiency. By encoding this combined route variable, we preserve important interaction information between origin and destination that would be lost if they were treated independently.

# Modeling Pipelines

### Visualization

Our modeling sequence ensures data integrity and prevents leakage by fitting all transformations on training data.

<img src="https://i.imgur.com/y0ukDz0.png" width="900">

### Data Preprocessing & Input Features

We utilize 17 input features across five families in our feature set.

#### Baseline Features (14 variables)

| Family | Features | Count |
|---|---|---|
| Route & Schedule | Flight Distance, Scheduled Departure Hour | 2 |
| Weather | Dry Bulb Temperature, Dew Point, Wind Speed, Wind Gusts, Visibility, Precipitation, Relative Humidity | 7 |
| Carrier & Airport | Airline Carrier, Origin Airport | 2 |
| Temporal | Day of the Week, Month | 2 |
| Target | Departure Delay Indicator | 1 |

### Engineered Features (replaces 7 variables)

| Raw Feature Area | Transformation | Engineered Feature Area | Rationale |
|---|---|---|---|
| Precipitation | Binary | Precipitation Indicator | Mostly zero, presence matters more than amount |
| Relative Humidity | Normalization | Scaled Humidity | Approximately normal, standardized for model convergence |
| Temperature | Normalization | Scaled Temperature | Approximately normal, standardized for model convergence |
| Wind Speed | Log-transform | Scaled Wind Speed | Right-skewed, log normalizes distribution |
| Visibility | Bucketed | Visibility Category | Left-skewed with a cap, bins for operational thresholds |
| Day of Week | Binary | Weekend Indicator | Weekend vs. weekday delay pattern |
| Origin + Destination | Concatenation | Flight Route | Captures route-specific delay tendencies |

### RFM Feature Set (adds 3 time-based features)

| Feature | RFM Type | Description |
|---|---|---|
| Origin Delay Frequency | Frequency | Share of flights delayed at origin airport in the last 7 days |
| Carrier Delay Frequency | Frequency | Share of flights delayed by carrier in the last 7 days |
| Origin Days Since Delay | Recency | Days since the latest delay at this origin airport |

All three features use backward-looking window functions to calculate frequency and recency, ensuring no information leaks.

---

### Loss Functions

**Logistic Regression** — log-loss with L2 regularization:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1 - y_i)\log(1 - \hat{p}_i)\right] + \lambda \|\mathbf{w}\|_2^2$$

**Random Forest** — Gini impurity at each split:

$$\text{Gini}(t) = 1 - \sum_{k=0}^{1} p_k^2$$

where $p_k$ is the proportion of class $k$ at node $t$.

---

# Results and Discussion of Results

# Conclusion

This project focuses on predicting flight delays to support proactive decision-making for XYZ Airline’s Operations Control (OCC) team. Flight delays are critical because they create cascading disruptions across crew scheduling, aircraft rotations, and passenger connections, leading to increased operational costs and reduced customer satisfaction.

We hypothesized that a machine learning pipeline incorporating weather, temporal, and operational features can accurately forecast flight delays at least two hours prior to departure. In this phase, we developed a scalable data pipeline, engineered key features, and implemented baseline and tree-based models using a time-aware validation framework. Our results show that Random Forest consistently outperforms Logistic Regression in recall, confirming that non-linear models better capture the complex drivers of flight delays.

- Discuss the significance of the results

These findings demonstrate that early delay prediction is both feasible and operationally valuable. In future work, we will enhance model performance through advanced feature engineering, neural networks, and hyperparameter tuning, with the goal of delivering a robust, production-ready system for real-world airline operations.

---

## Next Steps

- Improve model performance using **MLP neural networks and boosted models (LightGBM/GBT)**  
- Perform **hyperparameter tuning and threshold optimization** aligned with business cost  
- Expand feature engineering with **time-based, lag, and network (route-level) features**  
- Evaluate on a **true holdout dataset (e.g., 2019)** to assess generalization  
- Deliver an **optimal end-to-end pipeline** with actionable insights for OCC decision-making  

# Updated Credit assignment plan

### Phase 1 (Due 3/15)

#### Task List

| **Task**                                 | **Task Type** | **Effort Hours** | **Assigned To**          |
|------------------------------------------|---------------|------------------|--------------------------|
| Credit Assignment Plan                   | Report        |                2 | Daniel Yi                |
| Project Abstract                         | Report        |                2 | Frank Lin                |
| Data Description (1)                     | Report        |                1 | Frank Lin                |
| EDA for Weather Dataset                  | EDA           |                4 | Allen Y Li               |
| EDA for Airports + Airport Codes Dataset | EDA           |                2 | Frank Lin                |
| EDA for Flights Dataset                  | EDA           |                2 | Daniel Yi                |
| EDA for OTPW Dataset (1)                 | EDA           |                6 | William Shu              |
| EDA for OTPW Dataset (2)                 | EDA           |                3 | Jason Chang              |
| Summary EDA for All Datasets (1)         | Report        |                1 | Frank Lin                |
| Summary EDA for All Datasets (2)         | Report        |                1 | Jason Chang              |
| Machine Learning Algorithm Discussion (1)| Modeling      |                2 | Allen Y Li               |
| Machine Learning Algorithm Discussion (2)| Modeling      |                2 | Frank Lin                |
| Metrics Discussion (1)                   | Report        |                1 | Frank Lin                |
| Metrics Discussion (2)                   | Report        |                1 | William Shu              |
| Gantt Chart                              | Report        |                1 | Daniel Yi                |
| Pipeline Description (1)                 | Modeling      |                1 | Jason Chang              |
| Pipeline Description (2)                 | Modeling      |                1 | William Shu              |
| Team Member Emails                       | Report        |              0.5 | Daniel Yi                |
| Feature Engineering Plan (1)             | Report        |              0.5 | Daniel Yi                |
| Open Problems + Next Steps (1)           | Report        |                1 | Frank Lin                |
| Report Finalization                      | Report        |                2 | Frank Lin                |

#### Cumulative Hours

| Name        | Total Hours |
|-------------|-------------|
| Daniel Yi   | 7           |
| Frank Lin   | 12          |
| Jason Chang | 5           |
| William Shu | 7           |
| Allen Li    | 6           |


### Phase 2 (Due 4/5)

#### Task List

| Task                                                                                                                       | Task Type         | Effort Hours | Assigned To |
|----------------------------------------------------------------------------------------------------------------------------|-------------------|--------------|-------------|
| EDA for Weather Dataset (Cont.)                                                                                            | EDA               |            0 | Allen Y Li  |
| EDA for Airports + Airport Codes Dataset (Cont.)                                                                           | EDA               |            1 | Frank Lin   |
| EDA for Flights Dataset (Cont.)                                                                                            | EDA               |            0 | Jason Chang |
| EDA for OTPW Dataset (Cont.)                                                                                               | EDA               |            0 | William Shu |
| Joining Data (Extra Credit)                                                                                                | Modeling          |            0 | William Shu |
| Feature Engineering + Feature Transformation + Discussion                                                                  | Report + Modeling |            5 | Daniel Yi   |
| Model Three-month OTPW Dataset and perform experiments                                                                     | Modeling          |            1 | William Shu |
| Model One-year OTPW Dataset and perform experiments                                                                        | Modeling          |            3 | William Shu |
| Model Experiment Discussion + Cross Validation                                                                             | Report            |            3 | William Shu |
| Create new features that are highly predictive: at least one time-based feature, e.g., recency, frequency, monetary, (RFM) | Report + Modeling |            2 | Daniel Yi   |
| Team and project meta information.                                                                                         | Report            |            1 | Daniel Yi   |
| Discuss Modeling piplines and features                                                                                     | Report            |            0 | Jason Chang |
| Updated Credit Assignment Plan                                                                                             | Report            |            1 | Daniel Yi   |
| Updated Abstract                                                                                                           | Report            |            0 | Frank Lin   |
| EDA                                                                                                                        | Report            |            0 | Jason Chang |
| Project Description                                                                                                        | Report            |            0 | Allen Y Li  |
| Updated Conclusion + Next Steps                                                                                            | Report            |            0 | Frank Lin   |
| In-class Presentation (1)                                                                                                  | Presentation      |            0 | Allen Y Li  |
| In-class Presentation (2)                                                                                                  | Presentation      |            2 | Frank Lin   |
| In-class Presentation (3)                                                                                                  | Presentation      |            2 | William Shu |
| In-class Presentation (4)                                                                                                  | Presentation      |            2 | Daniel Yi   |
| In-class Presentation (5)                                                                                                  | Presentation      |            0 | Jason Chang |

#### Cumulative Hours

| Name        | Total Hours |
|-------------|-------------|
| Daniel Yi   |           0 |
| Frank Lin   |           0 |
| Jason Chang |           0 |
| William Shu |           0 |
| Allen Li    |           0 |


### Phase 3 (Due 4/5)

#### Task List

| **Task**                                                                                | **Task Type**     | **Effort Hours** | **Assigned To** |
|-----------------------------------------------------------------------------------------|-------------------|------------------|-----------------|
| Consider more sophisticated models (GBDT, MLP neural networks, etc.)                    | Report + Modeling |                  | TBD             |
| Fine-tune your pipeline using a grid search, or any search techniques                   | Report + Modeling |                  | TBD             |
| Do experiments on ALL the data for new features and experimental settings               | Report + Modeling |                  | TBD             |
| Feature engineering and refinement (at least one Time-based feature and Graph feature)  | Report + Modeling |                  | TBD             |
| Exciting novel directions that you pursued                                              | Report + Modeling |                  | TBD             |
| Final Abstract                                                                          | Report            |                  | TBD             |
| Data and Feature Engineering (EDA)                                                      | Report            |                  | TBD             |
| Neural Network (MLP)                                                                    | Report            |                  | TBD             |
| Leakage                                                                                 | Report            |                  | TBD             |
| Modeling Pipeline                                                                       | Report            |                  | TBD             |
| Results and discussion of results + Gap Analysis of Best Model                          | Report            |                  | TBD             |
| Concusion                                                                               | Report            |                  | TBD             |
| Clean Code + Report Finalization                                                        | Report            |                  | TBD             |
| In-class Presentation                                                                   | Presentation      |                  | TBD             |

#### Cumulative Hours

| Name        | Total Hours |
|-------------|-------------|
| Daniel Yi   |           0 |
| Frank Lin   |           0 |
| Jason Chang |           0 |
| William Shu |           0 |
| Allen Li    |           0 |

### Gantt 

<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/Project Task List - Gantt Chart-1.png" width="900"><br>